In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/yasserh/wine-quality-dataset/WineQT.csv


In [2]:
df = pd.read_csv("/kaggle/input/datasets/yasserh/wine-quality-dataset/WineQT.csv")
df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,0
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,1
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,2
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,3
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1138,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6,1592
1139,6.8,0.620,0.08,1.9,0.068,28.0,38.0,0.99651,3.42,0.82,9.5,6,1593
1140,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5,1594
1141,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6,1595


In [3]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import math

In [4]:
x = df[df.columns.drop(["Id", "quality"])]
y = df["quality"]
x,y

(      fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
 0               7.4             0.700         0.00             1.9      0.076   
 1               7.8             0.880         0.00             2.6      0.098   
 2               7.8             0.760         0.04             2.3      0.092   
 3              11.2             0.280         0.56             1.9      0.075   
 4               7.4             0.700         0.00             1.9      0.076   
 ...             ...               ...          ...             ...        ...   
 1138            6.3             0.510         0.13             2.3      0.076   
 1139            6.8             0.620         0.08             1.9      0.068   
 1140            6.2             0.600         0.08             2.0      0.090   
 1141            5.9             0.550         0.10             2.2      0.062   
 1142            5.9             0.645         0.12             2.0      0.075   
 
       free su

In [5]:
xtrain, xtest, ytrain, ytest = train_test_split(x,y, random_state=42, test_size=0.2)

In [6]:
np.random.seed(43)
a = np.full(len(xtrain), 0)
b = np.random.rand()

In [7]:
gama = 1

In [8]:
k = lambda xt,x : np.exp(-gama*np.sum((xt-x)**2, axis=1))
k(xtrain, xtest.iloc[0])

12       0.000000e+00
758      3.038290e-10
636      0.000000e+00
1109     8.970433e-66
743      0.000000e+00
            ...      
1044    2.160352e-297
1095     0.000000e+00
1130    6.233638e-122
860      0.000000e+00
1126     3.129512e-68
Length: 914, dtype: float64

In [9]:
z = lambda xt, x, yt : sum(a * yt * k(xt,x))
z(xtrain, xtest.iloc[0], ytrain)

0.0

In [10]:
yt = lambda t, y : 1 if t==y else -1

In [11]:
yp = lambda xt, x, yt: 1 if z(xt,x,yt)>0 else -1

In [12]:
da = lambda xt,x,yj : -ytrain * yj * k(xt,x)
da(xtrain, xtest.iloc[0], yt(6,ytest.iloc[0]))

12       0.000000e+00
758      1.822974e-09
636      0.000000e+00
1109     4.485216e-65
743      0.000000e+00
            ...      
1044    8.641407e-297
1095     0.000000e+00
1130    3.740183e-121
860      0.000000e+00
1126     1.877707e-67
Length: 914, dtype: float64

In [13]:
def train(yt, learning_rate=0.01):
    global a
    greda = np.sum(xtrain.apply(lambda x : da(xtrain, x, yt.loc[x.name]), axis=1), axis=0)
    a -= greda * learning_rate

In [14]:
# loss = lambda x, y : sum(x.apply(lambda xi : max(0,yp(xtrain, xi,yt(y,ytrain.loc[xi.name]))), axis=1))

In [15]:
# loss(xtrain,ytrain)

In [16]:
train(ytrain.apply(lambda y:yt(6,y)))
a

12      0
758     0
636     0
1109    0
743     0
       ..
1044    0
1095    0
1130    0
860     0
1126    0
Length: 914, dtype: int64

In [17]:
np.unique(a)

array([0])